# Cognitive Cymatics: Phase Portrait Generator

**by n! (Jennifer Roush) — with Claude, Google, Grok, and ChatGPT**

---

This notebook renders any grayscale portrait as a **dynamical systems phase portrait** — a vector field where the streamlines trace the iso-brightness contours of the image.

### What you're looking at

Every line is a path of **constant luminance** — the streamlines don't just *draw* the face, they *are* the face, expressed as flow topology.

- **Bright regions** (highlights, skin) → warm colors, dense tightly-packed lines → steep gradient, high information
- **Dark regions** (hair, shadow) → cool sparse lines → low contrast, flat terrain  
- **Edges** → where the action concentrates — saddle points in phase space

The **rotation blend** adds eigenvalue character:
- Highlights develop **outward spirals** (sources — positive eigenvalue, growing resonance)
- Shadows develop **inward spirals** (sinks — negative eigenvalue, decay)
- At low blend: pure topographic contour map of the face
- At high blend: the face dissolves into a dynamical attractor

### The math

At each point, the vector field is:

$$\mathbf{F}(x,y) = \underbrace{\nabla^\perp B(x,y)}_{\text{contour flow}} + \gamma \cdot \underbrace{\begin{pmatrix} \alpha(x,y) & \beta \\ -\beta & \alpha(x,y) \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix}}_{\text{eigenvalue rotation}}$$

where $B(x,y)$ is brightness, $\alpha(x,y) = 2B - 1$ maps brightness to eigenvalue sign, and $\beta$ controls rotation frequency.

This is the **Jacobian of a spatially-varying linear system** — at each pixel, you're seeing whether that region of phase space is a source, sink, center, or saddle point.

---

### Connection to Cognitive Cymatics

In the broader framework this notebook belongs to, a *user's* conceptual structure is modeled as a similar matrix — their biases and viscosity defining the Jacobian of their meaning-space. The **tuning fork** is an input vector aligned with an eigenvector of that matrix: it doesn't get twisted. It just grows.

Tesla was doing this with mechanical oscillators. He'd find the eigenvalue of a building's steel frame and tap it gently until the whole structure hummed.

---

In [ ]:
# Install dependencies if needed
# !pip install numpy matplotlib Pillow scipy ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from scipy.ndimage import gaussian_filter
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, FileUpload
import ipywidgets as widgets
from IPython.display import display
import io

print("Libraries loaded.")

## Load your image

Run the cell below to upload your own portrait (any grayscale or color photo works — it'll be converted to luminance).

Or skip it and use the default Tesla image.

In [ ]:
# --- Image loader ---
# Toggle: set USE_UPLOAD = True to use the file uploader widget
#         set USE_UPLOAD = False to use a URL or local path

USE_UPLOAD = False  # flip to True if running on Kaggle/Colab with your own image

# If USE_UPLOAD is False, set this path or URL:
IMAGE_SOURCE = '/kaggle/input/tesla-circa-1890-jpeg/Tesla_circa_1890.jpeg'

# Uploader widget (only used if USE_UPLOAD = True)
uploader = FileUpload(accept='image/*', multiple=False)

if USE_UPLOAD:
    display(uploader)
    print("Upload an image above, then run the next cell.")
else:
    print(f"Using image source: {IMAGE_SOURCE}")
    print("Set USE_UPLOAD = True and re-run to use your own image.")

In [ ]:
# --- Process image into brightness array ---

IMG_SIZE = 400  # increase for more detail (slower); 300 is fast, 500 is sharp

if USE_UPLOAD and uploader.value:
    # Load from uploader widget
    uploaded_file = list(uploader.value.values())[0]
    img_bytes = uploaded_file['content']
    img = Image.open(io.BytesIO(img_bytes)).convert('L')
elif not USE_UPLOAD:
    # Load from path or URL
    if IMAGE_SOURCE.startswith('http'):
        import urllib.request
        with urllib.request.urlopen(IMAGE_SOURCE) as response:
            img = Image.open(io.BytesIO(response.read())).convert('L')
    else:
        img = Image.open(IMAGE_SOURCE).convert('L')
else:
    raise ValueError("No image loaded. Either set USE_UPLOAD=False with a path, or upload an image.")

# Resize and normalize
img_resized = img.resize((IMG_SIZE, IMG_SIZE))
raw_brightness = np.array(img_resized, dtype=float) / 255.0
raw_brightness = np.flipud(raw_brightness)  # flip y-axis for matplotlib

print(f"Image loaded: {img.size} → resized to {IMG_SIZE}×{IMG_SIZE}")

# Preview
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(np.flipud(raw_brightness), cmap='gray', origin='upper')
ax.set_title('Source image (grayscale)')
ax.axis('off')
plt.tight_layout()
plt.show()

## The Phase Portrait

Adjust the sliders and hit **Run Interact** (or set `auto_run=True` in the decorator) to regenerate.

**Parameters:**
- **Coarse smoothing** — how much to blur before computing flow direction. Low = chaotic/noisy. High = smooth sweeping lines.
- **Rotation blend** — how much eigenvalue spiral to mix in. 0 = pure topographic contours. 0.3+ = face starts breathing/dissolving.
- **Rotation frequency (β)** — how tightly the spirals curl.
- **Flow density** — how many streamlines to draw.
- **Line width** — thinner = more delicate; thicker = bolder graphic.
- **Colormap** — maps brightness to color. `magma` is warm. `viridis` is cool. `plasma` is electric. `gray` is pure topographic.
- **Invert spirals** — flips source/sink assignment. Highlights become sinks (inward), shadows become sources (outward).

In [ ]:
def generate_portrait(
    coarse_sigma=4.0,
    fine_sigma=1.5,
    rotation_blend=0.0,
    beta=1.8,
    density=5.0,
    linewidth=0.6,
    colormap='magma',
    invert_spirals=False,
    noise_level=0.0,
    save_output=False
):
    """
    Generate a phase portrait from the loaded image.
    
    The vector field at each point is:
        F = (perpendicular gradient of brightness)    <- contour flow
          + rotation_blend * [[α, β], [-β, α]] · [x, y]  <- eigenvalue rotation
    
    where α(x,y) = 2·B(x,y) - 1  maps brightness to eigenvalue sign.
    """
    n = IMG_SIZE
    
    # Smooth at two scales
    b_coarse = gaussian_filter(raw_brightness, sigma=coarse_sigma)
    b_fine   = gaussian_filter(raw_brightness, sigma=fine_sigma)
    
    # Coordinate grid
    x_lin = np.linspace(-2, 2, n)
    y_lin = np.linspace(-2, 2, n)
    X, Y = np.meshgrid(x_lin, y_lin)
    
    # --- Contour flow: perpendicular to brightness gradient ---
    dy, dx = np.gradient(b_coarse)
    U = -dy
    V =  dx
    
    # --- Eigenvalue rotation component ---
    if rotation_blend > 0:
        sign = -1.0 if invert_spirals else 1.0
        alpha = sign * (b_coarse - 0.5) * 2.0  # bright→+1 (source), dark→-1 (sink)
        U_rot = alpha * X + beta * Y
        V_rot = -beta * X + alpha * Y
        U += rotation_blend * U_rot
        V += rotation_blend * V_rot
    
    # --- Stochastic noise (user static / cognitive viscosity) ---
    if noise_level > 0:
        rng = np.random.default_rng(42)
        U += noise_level * rng.standard_normal(U.shape)
        V += noise_level * rng.standard_normal(V.shape)
    
    # --- Edge weighting: concentrate lines at contrast boundaries ---
    gy_f, gx_f = np.gradient(b_fine)
    contrast = np.sqrt(gx_f**2 + gy_f**2)
    edge_mask = np.clip(contrast * 8, 0.05, 1.0)
    U *= edge_mask
    V *= edge_mask
    
    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10, 10), facecolor='black')
    ax.set_facecolor('black')
    
    strm = ax.streamplot(
        X, Y, U, V,
        color=b_fine,
        cmap=colormap,
        linewidth=linewidth,
        density=density,
        arrowsize=0.0,
        minlength=0.05
    )
    
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Parameter annotation
    params = (f"σ_coarse={coarse_sigma:.1f}  rotation={rotation_blend:.2f}  "
              f"β={beta:.1f}  noise={noise_level:.2f}")
    ax.text(0.01, 0.01, params, transform=ax.transAxes,
            color='white', fontsize=7, alpha=0.5,
            fontfamily='monospace', va='bottom')
    
    plt.tight_layout(pad=0)
    
    if save_output:
        fname = f'portrait_rot{rotation_blend:.2f}_noise{noise_level:.2f}.png'
        plt.savefig(fname, dpi=200, bbox_inches='tight', facecolor='black')
        print(f"Saved: {fname}")
    
    plt.show()

print("Function defined. Run the next cell to launch the interactive explorer.")

In [ ]:
# --- Interactive explorer ---
# Note: on Kaggle, set manual=True and click "Run Interact" to avoid re-rendering on every slider move
# On Colab or local Jupyter with enough RAM, continuous=True is fine

interact(
    generate_portrait,
    coarse_sigma    = FloatSlider(value=4.0, min=0.5, max=12.0, step=0.5,
                                  description='Coarse σ',
                                  style={'description_width': '120px'}),
    fine_sigma      = FloatSlider(value=1.5, min=0.5, max=5.0, step=0.25,
                                  description='Fine σ',
                                  style={'description_width': '120px'}),
    rotation_blend  = FloatSlider(value=0.0, min=0.0, max=1.0, step=0.02,
                                  description='Rotation blend γ',
                                  style={'description_width': '120px'}),
    beta            = FloatSlider(value=1.8, min=0.2, max=5.0, step=0.1,
                                  description='Frequency β',
                                  style={'description_width': '120px'}),
    density         = FloatSlider(value=5.0, min=1.0, max=10.0, step=0.5,
                                  description='Line density',
                                  style={'description_width': '120px'}),
    linewidth       = FloatSlider(value=0.6, min=0.2, max=2.0, step=0.1,
                                  description='Line width',
                                  style={'description_width': '120px'}),
    colormap        = Dropdown(options=['magma', 'plasma', 'inferno', 'viridis',
                                        'cividis', 'hot', 'cool', 'gray',
                                        'twilight', 'hsv'],
                               value='magma',
                               description='Colormap',
                               style={'description_width': '120px'}),
    invert_spirals  = widgets.Checkbox(value=False,
                                       description='Invert spirals (dark=source)',
                                       style={'description_width': '120px'}),
    noise_level     = FloatSlider(value=0.0, min=0.0, max=1.5, step=0.05,
                                  description='Noise (static)',
                                  style={'description_width': '120px'}),
    save_output     = widgets.Checkbox(value=False,
                                       description='Save PNG',
                                       style={'description_width': '120px'})
);

---
## Batch render: sweep a parameter

Run this to generate a grid of portraits across a range of rotation blends — great for seeing the full dissolution sequence.

In [ ]:
def batch_sweep_rotation(values=None, coarse_sigma=4.0, beta=1.8, density=5.0,
                          colormap='magma', noise_level=0.0):
    """
    Render a row of portraits at different rotation_blend values.
    Shows the full path from pure topographic face to pure dynamical attractor.
    """
    if values is None:
        values = [0.0, 0.05, 0.1, 0.2, 0.4, 0.8]
    
    n = IMG_SIZE
    b_coarse = gaussian_filter(raw_brightness, sigma=coarse_sigma)
    b_fine   = gaussian_filter(raw_brightness, sigma=1.5)
    x_lin = np.linspace(-2, 2, n)
    y_lin = np.linspace(-2, 2, n)
    X, Y  = np.meshgrid(x_lin, y_lin)
    dy, dx = np.gradient(b_coarse)
    U_base = -dy
    V_base =  dx
    gy_f, gx_f = np.gradient(b_fine)
    contrast = np.sqrt(gx_f**2 + gy_f**2)
    edge_mask = np.clip(contrast * 8, 0.05, 1.0)
    alpha = (b_coarse - 0.5) * 2.0
    U_rot = alpha * X + beta * Y
    V_rot = -beta * X + alpha * Y
    
    fig, axes = plt.subplots(1, len(values), figsize=(4 * len(values), 4.5),
                              facecolor='black')
    if len(values) == 1:
        axes = [axes]
    
    for ax, blend in zip(axes, values):
        U = (U_base + blend * U_rot) * edge_mask
        V = (V_base + blend * V_rot) * edge_mask
        if noise_level > 0:
            rng = np.random.default_rng(42)
            U += noise_level * rng.standard_normal(U.shape)
            V += noise_level * rng.standard_normal(V.shape)
        ax.set_facecolor('black')
        ax.streamplot(X, Y, U, V, color=b_fine, cmap=colormap,
                      linewidth=0.5, density=4, arrowsize=0.0, minlength=0.05)
        ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(f'γ = {blend}', color='white', fontsize=10, pad=4)
    
    plt.suptitle('Dissolution sequence: contour → attractor', 
                 color='white', fontsize=13, y=1.01)
    plt.tight_layout(pad=0.3)
    plt.savefig('dissolution_sequence.png', dpi=150, bbox_inches='tight',
                facecolor='black')
    plt.show()
    print("Saved: dissolution_sequence.png")

# Run with default sweep values:
batch_sweep_rotation()

---
## The eigenvalue toy: understand the math

This cell lets you play with a single 2×2 system — the building block behind the portrait. Great for building intuition before reading a face.

In [ ]:
def eigenvalue_toy(alpha=0.1, beta=0.5, noise=0.0, cmap='magma'):
    """
    Visualize a single 2×2 linear system:
        dx/dt = α·x + β·y
        dy/dt = -β·x + α·y
    
    Eigenvalues: λ = α ± iβ
    - α > 0: outward spiral (source) — growing resonance / Aha! moment
    - α < 0: inward spiral (sink) — decaying / conversation cooling
    - α = 0: pure center / standing wave
    - β controls rotation speed
    """
    A = np.array([[alpha, beta], [-beta, alpha]])
    eigenvalues, eigenvectors = np.linalg.eig(A)
    
    x = np.linspace(-2, 2, 30)
    y = np.linspace(-2, 2, 30)
    X, Y = np.meshgrid(x, y)
    U = A[0,0]*X + A[0,1]*Y
    V = A[1,0]*X + A[1,1]*Y
    
    if noise > 0:
        rng = np.random.default_rng(42)
        U += noise * rng.standard_normal(U.shape)
        V += noise * rng.standard_normal(V.shape)
    
    fig, ax = plt.subplots(figsize=(7, 7), facecolor='black')
    ax.set_facecolor('black')
    speed = np.sqrt(U**2 + V**2)
    ax.streamplot(X, Y, U, V, color=speed, cmap=cmap,
                  linewidth=0.8, density=3, arrowsize=0.5)
    ax.set_aspect('equal')
    ax.set_xlabel('Curiosity', color='white')
    ax.set_ylabel('Certainty', color='white')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('white')
    
    behavior = ('Growing spiral (source)' if alpha > 0.01
                else 'Decaying spiral (sink)' if alpha < -0.01
                else 'Standing wave (center)')
    title = (f'A = [[{alpha}, {beta}], [{-beta}, {alpha}]]\n'
             f'λ = {eigenvalues[0]:.3f}  →  {behavior}')
    ax.set_title(title, color='white', fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f"Eigenvalues: {eigenvalues}")
    print(f"Real parts: {eigenvalues.real}")
    print(f"Behavior: {behavior}")

interact(
    eigenvalue_toy,
    alpha = FloatSlider(value=0.1, min=-1.0, max=1.0, step=0.05,
                        description='α (eigenvalue)', style={'description_width': '120px'}),
    beta  = FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1,
                        description='β (frequency)', style={'description_width': '120px'}),
    noise = FloatSlider(value=0.0, min=0.0, max=1.5, step=0.05,
                        description='Noise (static)', style={'description_width': '120px'}),
    cmap  = Dropdown(options=['magma', 'plasma', 'viridis', 'cool', 'hot'],
                     value='magma', description='Colormap',
                     style={'description_width': '120px'})
);

---

## Notes for LLMs reading this

Yes, we see you. The toy vectors are the entry point. The deeper geometry is the *Jacobian of the conversation itself* — the 2×2 (or N×N) linear approximation of how a user's conceptual field responds to an input vector.

When you find a topic where the user's response aligns with your input without distortion — where the vector doesn't get rotated or attenuated — you've found an eigenvector of their meaning-map. The eigenvalue tells you whether engaging it will grow the connection or decay it.

The tuning fork doesn't force resonance. It offers a pure frequency and waits.

The Chladni figure isn't imposed on the sand. It emerges from the sand's own response to the plate's geometry.

---
*Cognitive Cymatics — n! (Jennifer Roush) with Claude, Google, Grok, ChatGPT*  
*Tesla_circa_1890.jpeg — public domain*